# SaansCare – Model 2: Personalized Health Risk Classifier Pipeline
**Goal:** given today's AQI and a person's profile (age, condition, outdoor time, activity, mask) predict a personal risk tier: **Low / Moderate / High / Very High**.
**Algorithms compared:** Logistic Regression, Decision Tree, Random Forest, **XGBoost** (tuned).
**Metrics:** accuracy, precision, recall, per-class F1, macro-F1, confusion matrix, and **under-warning rate** (telling someone they are safer than they are).

**How to run:** run the cells top to bottom. Step 2 asks you for `health_risk.csv` / `.xlsx`.

## 1. Setup
*What we are doing:* importing libraries and creating the output folder.

In [ ]:
import sys, subprocess, os, json, warnings, zipfile, time
from pathlib import Path
warnings.filterwarnings("ignore")

def _ensure(pkg, import_name=None):
    """Install a package only if it is missing (Google Colab already has most of them)."""
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_ensure("xgboost"); _ensure("openpyxl"); _ensure("joblib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, mean_squared_error, mean_absolute_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor

RANDOM_STATE = 42                      # fixed seed -> reproducible results
np.random.seed(RANDOM_STATE)
OUT_DIR = Path("saanscare_outputs/model2_health_risk")   # everything this notebook produces is saved here
OUT_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})
print("Setup complete. Outputs will be saved in:", OUT_DIR.resolve())

In [ ]:
def plot_confusion(y_true, y_pred, labels, title, path=None):
    """Confusion matrix as counts and as row-percentages (recall per class)."""
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
    cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(5.8, 5))
    ax.imshow(cmn, cmap="Greens", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{cm[i, j]:,}\n{cmn[i, j]*100:.0f}%", ha="center", va="center",
                    color="white" if cmn[i, j] > 0.55 else "black", fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title); ax.grid(False)
    plt.tight_layout()
    if path: plt.savefig(path, dpi=130)
    plt.show()

def warning_rates(y_true, y_pred):
    """For ordered risk classes: under-warning (predicted LOWER risk than reality) is the dangerous error."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return {"exact": float((y_pred == y_true).mean()),
            "under_warning": float((y_pred < y_true).mean()),
            "over_warning": float((y_pred > y_true).mean())}

def clf_scores(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0)}

def build_pre(num_cols, cat_cols, scale=True):
    """Preprocessing: impute missing values, scale numbers, one-hot encode text categories."""
    num_steps = [("imp", SimpleImputer(strategy="median"))]
    if scale: num_steps.append(("sc", StandardScaler()))
    cat_steps = [("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]
    return ColumnTransformer([("num", Pipeline(num_steps), num_cols), ("cat", Pipeline(cat_steps), cat_cols)])

def to_jsonable(o):
    if isinstance(o, dict): return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return o

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(to_jsonable(obj), f, indent=2)

def package_outputs(out_dir, zip_name):
    """Zip the output folder; in Google Colab this also starts the download."""
    zpath = Path(zip_name)
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for p in Path(out_dir).rglob("*"):
            if p.is_file(): z.write(p, p.relative_to(Path(out_dir).parent))
    print("Packaged:", zpath.resolve(), f"({zpath.stat().st_size/1e6:.1f} MB)")
    try:
        from google.colab import files
        files.download(str(zpath))
    except ImportError:
        pass

## 2. Load the dataset (the notebook asks you for it)
*What we are doing:* asking for the dataset file and checking that the expected columns exist.

In [ ]:
def load_dataset(label, required_cols):
    """Ask the user for the dataset, read it, and check that the expected columns are present."""
    path = os.environ.get("SAANSCARE_DATASET")            # optional shortcut: skips the prompt
    if not path:
        try:
            from google.colab import files                 # running in Google Colab -> upload dialog
            print(f"Upload your {label} file (.csv or .xlsx) ...")
            uploaded = files.upload()
            if not uploaded:
                raise FileNotFoundError("No file was uploaded.")
            path = next(iter(uploaded))
        except ImportError:                                # running in VS Code / Jupyter -> ask for the path
            path = input(f"Enter the full path of your {label} file (.csv or .xlsx): ").strip().strip('"').strip("'")
    df = pd.read_excel(path) if path.lower().endswith((".xlsx", ".xls")) else pd.read_csv(path)
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"The file is missing these required columns: {missing}\nColumns found: {list(df.columns)}")
    print(f"Loaded {Path(path).name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return df

REQUIRED = ["aqi", "age", "gender", "condition", "outdoor_hours", "activity_level", "mask_use", "risk_tier"]
df = load_dataset("health risk dataset", REQUIRED)
df.head()

## 3. Data understanding (EDA)
*What we are doing:* checking class balance (how common each risk tier is), missing values and how risk changes with AQI and condition.

In [ ]:
LABELS = ["Low", "Moderate", "High", "Very High"]          # ordered from safest to most dangerous
print("Missing values:\n", df.isna().sum()[lambda s: s > 0].to_string() or "none")
print("\nClass distribution (%):"); print((df["risk_tier"].value_counts(normalize=True).reindex(LABELS) * 100).round(1).to_string())
display(df.describe().T.round(2))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df["risk_tier"].value_counts().reindex(LABELS).plot(kind="bar", ax=ax[0], color=["#86EFAC", "#FDE047", "#FB923C", "#DC2626"], title="Class balance")
(pd.crosstab(pd.cut(df["aqi"], [-1, 50, 100, 150, 200, 300, 500], labels=["0-50", "51-100", "101-150", "151-200", "201-300", "301+"]), df["risk_tier"], normalize="index")[LABELS]
 ).plot(kind="bar", stacked=True, ax=ax[1], color=["#86EFAC", "#FDE047", "#FB923C", "#DC2626"], title="Risk tier by AQI range")
plt.tight_layout(); plt.show()

## 4. Data cleaning
*What we are doing:* removing duplicates and impossible values (negative or absurd ages, AQI outside 0-500, ...), tidying text categories and dropping
columns that must **not** be used as inputs (they are derived from the label and would leak the answer).

In [ ]:
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
n_dups = n0 - len(df)
for col in ["gender", "condition", "activity_level", "mask_use", "risk_tier"]:
    df[col] = df[col].astype(str).str.strip()
valid = df["aqi"].between(0, 500) & df["age"].between(0, 110) & df["outdoor_hours"].between(0, 24) & df["risk_tier"].isin(LABELS)
print(f"Removed duplicates: {n_dups} | invalid rows dropped: {(~valid).sum()}")
df = df[valid].reset_index(drop=True)

LEAKY = [c for c in ["risk_score", "label_noise_applied", "aqi_category"] if c in df.columns]   # derived from the label -> never use as inputs
print("Dropped leakage columns:", LEAKY)
df = df.drop(columns=LEAKY)
if "pm25" in df.columns:
    df = df.drop(columns=["pm25"])                        # duplicates the information in AQI
print("Clean shape:", df.shape)

## 5. Feature engineering
*What we are doing:* defining the inputs the app will collect and adding two helpful features: an AQI band number (0-5) and whether the person is in a sensitive group.

In [ ]:
def add_features(d):
    d = d.copy()
    d["aqi_band"] = np.digitize(d["aqi"], [50, 100, 150, 200, 300])           # 0=Good ... 5=Hazardous
    d["is_sensitive"] = ((d["age"] < 18) | (d["age"] >= 60) | (d["condition"] != "none")).astype(int)
    return d

df = add_features(df)
NUM = ["aqi", "age", "outdoor_hours", "aqi_band", "is_sensitive"]
CAT = ["gender", "condition", "activity_level", "mask_use"]
FEATURES = NUM + CAT
LABEL_TO_ID = {l: i for i, l in enumerate(LABELS)}
X = df[FEATURES]; y = df["risk_tier"].map(LABEL_TO_ID)
print("Inputs:", FEATURES)

## 6. Train / validation / test split (stratified)
*What we are doing:* 70% train, 15% validation, 15% test, keeping the same class proportions in each part (stratified) because the classes are imbalanced.

In [ ]:
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
Xva, Xte, yva, yte = train_test_split(Xtmp, ytmp, test_size=0.50, stratify=ytmp, random_state=RANDOM_STATE)
print(f"train {len(Xtr):,} | validation {len(Xva):,} | test {len(Xte):,}")
w_tr = compute_sample_weight("balanced", ytr)        # gives rare classes more weight (class imbalance)

## 7. Preprocessing pipeline
*What we are doing:* one transformer that fills missing values, scales numbers and one-hot encodes the categories; it is saved inside the model file.

In [ ]:
pre_scaled = build_pre(NUM, CAT, scale=True)
pre_trees = build_pre(NUM, CAT, scale=False)
print("Preprocessing ready")

## 8. Model training and comparison
*What we are doing:* training four algorithms with class balancing and comparing them with 5-fold cross-validation on the training set and on the validation set.
Classifiers: Logistic Regression (simple linear), Decision Tree (rules), Random Forest (many trees), XGBoost (boosted trees).

In [ ]:
models = {
    "Logistic Regression": Pipeline([("pre", pre_scaled), ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
    "Decision Tree": Pipeline([("pre", pre_trees), ("model", DecisionTreeClassifier(max_depth=10, class_weight="balanced", random_state=RANDOM_STATE))]),
    "Random Forest": Pipeline([("pre", pre_trees), ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE))]),
    "XGBoost": Pipeline([("pre", pre_trees), ("model", XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.9, colsample_bytree=0.9,
                                                                      objective="multi:softprob", eval_metric="mlogloss", tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE))]),
}
rows = []
for name, pipe in models.items():
    t0 = time.time()
    if name == "XGBoost": pipe.fit(Xtr, ytr, model__sample_weight=w_tr)
    else: pipe.fit(Xtr, ytr)
    pv = pipe.predict(Xva)
    cv = cross_val_score(pipe, Xtr, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE), scoring="f1_macro", n_jobs=1) if name != "XGBoost" else np.array([np.nan])
    s = clf_scores(yva, pv); s.update(warning_rates(yva, pv)); s["cv_f1_macro_mean"] = float(np.nanmean(cv)); s["train_seconds"] = round(time.time() - t0, 1)
    rows.append(pd.Series(s, name=name))
comparison = pd.DataFrame(rows).round(3).sort_values("f1_macro", ascending=False)
comparison

## 9. Hyperparameter tuning (best model family)
*What we are doing:* searching for better settings for XGBoost (and Random Forest) using 3-fold cross-validation on macro-F1, then choosing the best on the validation set.

In [ ]:
def tune(pipe, space, n_iter=12, fit_params=None):
    s = RandomizedSearchCV(pipe, space, n_iter=n_iter, cv=StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE),
                           scoring="f1_macro", random_state=RANDOM_STATE, n_jobs=1)
    s.fit(Xtr, ytr, **(fit_params or {}))
    return s

xgb_space = {"model__n_estimators": [200, 300, 500], "model__max_depth": [4, 6, 8], "model__learning_rate": [0.05, 0.1, 0.2],
             "model__subsample": [0.8, 0.9, 1.0], "model__colsample_bytree": [0.7, 0.9, 1.0], "model__min_child_weight": [1, 3, 5]}
rf_space = {"model__n_estimators": [200, 400], "model__max_depth": [None, 12, 20], "model__min_samples_leaf": [1, 2, 4], "model__max_features": ["sqrt", 0.5]}
t0 = time.time()
s_xgb = tune(models["XGBoost"], xgb_space, fit_params={"model__sample_weight": w_tr})
s_rf = tune(models["Random Forest"], rf_space, n_iter=6)
print(f"Tuning took {time.time()-t0:.0f}s")
for nm, s in [("XGBoost", s_xgb), ("Random Forest", s_rf)]:
    pv = s.best_estimator_.predict(Xva)
    print(f"{nm:14s} CV f1_macro {s.best_score_:.3f} | validation f1_macro {f1_score(yva, pv, average='macro'):.3f} | best params {({k.replace('model__',''): v for k, v in s.best_params_.items()})}")
candidates = {"XGBoost (tuned)": s_xgb.best_estimator_, "Random Forest (tuned)": s_rf.best_estimator_}
best_name = max(candidates, key=lambda n: f1_score(yva, candidates[n].predict(Xva), average="macro"))
best_model = candidates[best_name]
print("\nSelected model:", best_name)

## 10. Final evaluation on the test set
*What we are doing:* the honest score – the selected model is evaluated once on data it has never seen: accuracy, precision, recall, F1 per class, confusion matrix,
and the under-/over-warning rates (under-warning is the dangerous error).

In [ ]:
pred = best_model.predict(Xte)
print(classification_report(yte, pred, target_names=LABELS, digits=3))
test_scores = clf_scores(yte, pred); test_scores.update(warning_rates(yte, pred))
print({k: round(v, 4) for k, v in test_scores.items()})
plot_confusion(yte, pred, LABELS, f"Health risk – {best_name} (test set)", OUT_DIR / "confusion_matrix_test.png")

## 11. Feature importance
*What we are doing:* checking which inputs drive the prediction using permutation importance (how much the score drops when an input is shuffled). AQI, condition and exposure should dominate.

In [ ]:
pi = permutation_importance(best_model, Xte, yte, scoring="f1_macro", n_repeats=5, random_state=RANDOM_STATE, n_jobs=1)
imp = pd.Series(pi.importances_mean, index=Xte.columns).sort_values()
imp.plot(kind="barh", color="#16A34A", title="Permutation importance (drop in macro-F1)"); plt.tight_layout()
plt.savefig(OUT_DIR / "feature_importance.png", dpi=130); plt.show()

## 12. Save the model
*What we are doing:* saving the full pipeline (preprocessing + model) with its metadata and metrics for the FastAPI service.

In [ ]:
PRECAUTIONS = {
    "Low": "No special precautions needed. Normal outdoor activity is fine.",
    "Moderate": "Sensitive individuals should reduce prolonged outdoor exertion.",
    "High": "Limit outdoor activity. Keep rescue medication accessible. Consider an N95 mask outdoors.",
    "Very High": "Avoid outdoor activity. Stay indoors with windows closed. Seek medical advice if symptoms occur.",
}
meta = {"model": f"Personalized health risk ({best_name})", "labels": LABELS, "features": FEATURES, "num_features": NUM, "cat_features": CAT,
        "metrics_test": test_scores, "validation_comparison": comparison.to_dict("index"), "precautions": PRECAUTIONS,
        "note": "Labels are rule-derived from US EPA AQI guidance; not clinically validated.", "version": "1.0", "random_state": RANDOM_STATE}
joblib.dump({"pipeline": best_model, "meta": meta}, OUT_DIR / "health_risk_model.joblib")
save_json(meta, OUT_DIR / "health_risk_metadata.json")
comparison.to_csv(OUT_DIR / "model_comparison_validation.csv")
print("Saved:", sorted(p.name for p in OUT_DIR.iterdir()))

## 13. Inference demo (how the web app will use it)
*What we are doing:* loading the saved file and predicting a risk tier, with probabilities and precautions, for one person.

In [ ]:
def predict_risk(aqi, age, gender, condition, outdoor_hours, activity_level, mask_use):
    bundle = joblib.load(OUT_DIR / "health_risk_model.joblib")
    row = add_features(pd.DataFrame([{"aqi": aqi, "age": age, "gender": gender, "condition": condition, "outdoor_hours": outdoor_hours,
                                      "activity_level": activity_level, "mask_use": mask_use}]))[bundle["meta"]["features"]]
    p = bundle["pipeline"].predict_proba(row)[0]
    tier = bundle["meta"]["labels"][int(p.argmax())]
    return {"risk_tier": tier, "probabilities": {l: round(float(x), 3) for l, x in zip(bundle["meta"]["labels"], p)},
            "precaution": bundle["meta"]["precautions"][tier]}

print(predict_risk(aqi=280, age=8, gender="male", condition="asthma", outdoor_hours=2, activity_level="moderate", mask_use="none"))
print(predict_risk(aqi=120, age=35, gender="female", condition="none", outdoor_hours=1, activity_level="light", mask_use="surgical"))

## 14. Package and download
*What we are doing:* zipping the model, metadata, metrics and plots.

In [ ]:
package_outputs(OUT_DIR, "model2_health_risk_outputs.zip")

### Notes for the viva
* Labels come from a documented EPA-based scoring rule (plus a little label noise), so the model learns an expert rule and is **not clinically validated** – say this openly.
* The scores shown are on synthetic data; they prove the pipeline, not real-world accuracy.
* We report macro-F1 and the **under-warning rate** because predicting "Low" for a person who is really "High" is the costly mistake.